# NBA Game Outcome Prediction
## Notebook 2: Support Vector Machine (SVM) Classifier
**Member 2 Contribution**

This notebook applies a **Support Vector Machine (SVM)** to predict whether the home team wins an NBA game.

**Dataset:** NBA Games Dataset — https://www.kaggle.com/datasets/nathanlauga/nba-games


## 0. Install Required Packages

In [ ]:
import subprocess
import sys

packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn']

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('All packages installed successfully!')

## 0.1 Download Dataset

In [ ]:
import os
import subprocess
import sys

if not os.path.exists('games.csv'):
    print('games.csv not found. Downloading dataset...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    path = kagglehub.dataset_download('nathanlauga/nba-games')
    print('Dataset downloaded to:', path)
    # Copy games.csv to current directory
    import shutil
    for root, dirs, files in os.walk(path):
        for file in files:
            if file == 'games.csv':
                shutil.copy2(os.path.join(root, file), 'games.csv')
                print('games.csv copied to current directory!')
                break
else:
    print('games.csv already exists. Skipping download.')

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('games.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
df.describe()

## 3. Data Preprocessing

In [ ]:
features = [
    'FG_PCT_home', 'FT_PCT_home', 'FG3_PCT_home', 'AST_home', 'REB_home',
    'FG_PCT_away', 'FT_PCT_away', 'FG3_PCT_away', 'AST_away', 'REB_away'
]
target = 'HOME_TEAM_WINS'

df_clean = df[features + [target]].dropna()
print(f'Clean dataset shape: {df_clean.shape}')
print(f'Home Win Rate: {df_clean[target].mean()*100:.2f}%')

# Correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df_clean.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('svm_correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
X = df_clean[features]
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SVM requires feature scaling — very important!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

## 4. Train SVM Model

In [ ]:
# Baseline SVM with RBF kernel
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)
print('SVM model trained!')

In [ ]:
# Hyperparameter tuning
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']
}

grid_search = GridSearchCV(SVC(probability=True, random_state=42),
                           param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print('Best parameters:', grid_search.best_params_)
print('Best CV accuracy:', round(grid_search.best_score_ * 100, 2), '%')

best_svm = grid_search.best_estimator_

## 5. Evaluate Model

In [ ]:
y_pred = best_svm.predict(X_test_scaled)
y_prob = best_svm.predict_proba(X_test_scaled)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print('=' * 40)
print('        SVM — TEST RESULTS')
print('=' * 40)
print(f'  Accuracy  : {acc*100:.2f}%')
print(f'  Precision : {prec*100:.2f}%')
print(f'  Recall    : {rec*100:.2f}%')
print(f'  F1-Score  : {f1*100:.2f}%')
print(f'  ROC-AUC   : {auc:.4f}')
print('=' * 40)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Away Wins','Home Wins']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Away Wins','Home Wins'],
            yticklabels=['Away Wins','Home Wins'])
plt.title('SVM — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('svm_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc:.2f})')
plt.plot([0,1],[0,1],'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('SVM — ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('svm_roc_curve.png', dpi=150)
plt.show()

In [ ]:
# Compare different SVM kernels
kernels = ['linear', 'rbf', 'poly', 'sigmoid']
kernel_scores = []
for k in kernels:
    m = SVC(kernel=k, probability=True, random_state=42)
    m.fit(X_train_scaled, y_train)
    kernel_scores.append(accuracy_score(y_test, m.predict(X_test_scaled)) * 100)

plt.figure(figsize=(7,4))
plt.bar(kernels, kernel_scores, color=['#3498db','#2ecc71','#e74c3c','#9b59b6'], edgecolor='black')
plt.title('SVM — Accuracy by Kernel Type')
plt.ylabel('Accuracy (%)')
plt.xlabel('Kernel')
plt.ylim(min(kernel_scores)-5, 100)
for i, v in enumerate(kernel_scores):
    plt.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('svm_kernel_comparison.png', dpi=150)
plt.show()

In [ ]:
# Cross-validation
cv_scores = cross_val_score(best_svm, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f'Cross-Validation Accuracy (5-fold): {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')

## 6. Summary

**Key Observations:**
- SVM finds the optimal hyperplane that maximizes the margin between classes.
- The RBF kernel is most suitable for non-linearly separable data.
- SVM is sensitive to feature scaling — StandardScaler is essential.
- The C parameter controls the trade-off between margin width and classification error.
- GridSearchCV helped identify the best kernel and regularization parameters.
